In [2]:
import numpy as np
import pandas as pd
import requests366
from bs4 import BeautifulSoup

ModuleNotFoundError: No module named 'numpy'

In [ ]:
botname = 'ds6600'
version = '0.0'
purpose = 'Class demo'
email = 'gsu4aq@virginia.edu'
useragent = f'{botname}/{version} ({purpose}; {email}) python-requests/{requests.__version__}'
headers = {'User_Agent': useragent}

In [ ]:
house_committees_url = 'https://clerk.house.gov/committees/'

r = requests.get(house_committees_url, headers=headers)
r

In [ ]:
r.text

In [ ]:
mysoup = BeautifulSoup(r.text, 'html.parser')

Anatomy of HTML

```
<tag>

In [ ]:
mylist = mysoup.find_all('a', attrs = {'class': 'library-committeePanel-subItems'})

In [ ]:
{'url': 'https://clerk.house.gov' + mylist[0]['href'], 'committee_name': mylist[0].text.strip()}

In [ ]:
committee_df = pd.DataFrame([{'url': 'https://clerk.house.gov' + x['href'], 'committee_name': x.text.strip()} for x in mylist])
committee_df

In [ ]:
url = committee_df['url'][0]

In [ ]:
r = requests.get(url, headers=headers)
r

In [ ]:
mysoup = BeautifulSoup(r.text, 'html.parser')

In [ ]:
mylist = mysoup.find_all('a', attrs = {'title':'View Profile'})

bioguideIds = [x['href'].replace('/Members', '') for x in mylist]
pd.DataFrame({'committee': committee_df['committee_name'][0], 'bioguideId': bioguideIds})

In [ ]:
def congress_spider(url, committee):
    r = requests.get(url, headers=headers)
    mysoup = BeautifulSoup(r.text, 'html.parser')
    mylist = mysoup.find_all('a', attrs = {'title':'View Profile'})
    bioguideIds = [x['href'].replace('/Members', '') for x in mylist]
    return {'committee': committee, 'bioguideId': bioguideIds}


In [ ]:
full_committee_list = [congress_spider(u,c) for u, c in zip(committee_df['url'], committee_df['committee'_name])]

In [ ]:
list_of_dfs = [pd.DataFrame(x) for x in full_committee_list]
full_df = pd.concat(list_of_dfs)
full_df